# 만기와 연도를 더 잘게 나누면 놓친 차이가 보일까?

주 분석 이후에 생긴 세 가지 질문을 검토한다. 단기와 장기 차입의 관계가 다른지, 2022년과 2023년이 다른지, 초기 수익성과 업종 규모를 고려하면 결과가 달라지는지다.

ICR(이자보상배율)은 영업이익을 이자비용으로 나눈 값이다. 3배라면 영업이익이 그해 이자비용의 세 배라는 뜻이다. 한 행은 한 업종의 한 해다. 


[전체 질문 목록](../README.md)

**표 읽는 법**　N은 업종·연도 관측 수, G는 업종 수다. 회귀표의 beta는 추정한 관계의 크기, lo와 hi는 95% 신뢰구간의 양 끝이다. p값은 계수가 0이라는 가정에서 지금만큼 또는 더 극단적인 검정통계량이 나올 확률이다.

# 추가 가설·통제 검토 및 p값 설명 확장 v2
2026-11-03 사용자 승인: 전체 분석 정리에 통계 해석을 추가하고 근거 있는 추가 가설/통제를 실제 실행. 원본과 이전 분석 보존. 기존 결과를 본 뒤의 탐색적 확장이며 사전 확증 분석으로 표현하지 않는다.

## 회귀 실행 전 고정한 범위
2018–23, 40업종240행 전체 유지. 극단값 삭제·윈저화 없음. 사전2018–20 평균, 2021별도, Post22–23, 업종·연도FE, 원자재·환율·경기지수 추가 없음. 비용 대용률 및 공시 ICR 두 결과만 사용한다. 원 데이터의 집계 범위·부채 포함항목 미확인은 그대로다.

N1: 단기차입/자산 SD와 장기차입/자산 LD를 동시에 포함하고 L을 통제한다. 각 변수에 Post와2021항. 핵심 검정=βSDP−βLDP=0, 결과2개. SD=(단기차입+단기사채)/자산×100, LD=(장기차입+장기사채)/자산×100의 사전 연도비율 평균. 같은 총차입/자산에서 1%p를 장기에서 단기로 바꾼 구성 차이의 선형 대비다. 실제 만기·변동금리 노출이라고 단정하지 않는다.
N2: 기존 Q/S/L 기간항에서 Post를2022와2023으로 분리하고2021항 유지. 핵심 검정=βS2023−βS2022 및 βQ2023−βQ2022, 결과2개이므로4검정. 특정 해가 유의한지만 보고 둘이 다르다고 주장하지 않고 직접 차이를 검정한다. 업종별 경기·이익변화가 섞이므로 금리 지연전달의 인과 검정은 아니다.
N3: Q/S/L 기본 모형에 사전 영업이익률 P0=2018–20 margin평균과 규모 Z0=2018–20 log(자산)의 평균 각각×Post/×2021을 추가. 동시기 성과를 통제하는 대신 초기 수익성과 규모가 다른 업종의 변화 차이를 고려하는 연구자 제안. 핵심 검정=조건부 SP=0, 결과2개. 통제를 늘렸다는 이유로 더 타당한 모형이라고 선택하지 않는다. 초기 수익성에 대한 회귀평균·누락변수·측정오차 한계는 남는다.

총8개 핵심검정. CR1업종 군집/t39 CI, WCR-S31 귀무제약 Rademacher9999회 seed20261103, 원p와 Holm8보정 모두 보고. 양측검정, 이번 묶음만의 보정이며 전체 프로젝트의 사후 탐색을 해소하지 않음. N1/N2는 두 계수 차이의 공분산을 이용. bootstrap은 동일한 차이 계수가 직접 추정되도록 재매개화한다. 모든 통제계수도 공개하나 유의항만 후속 가설로 채택하지 않는다.

## 학술·공식 근거와 적용 한계
[ASA 2016 공식 발표문](https://www.amstat.org/asa/files/pdfs/p-valuestatement.pdf) 2쪽 원칙2: “P-values do not measure the probability that the studied hypothesis is true”. 해당 발표문 본문 원칙을 확인했으며 원 논문 전문을 읽었다는 주장이 아니다. p값·CI·크기·설계 한계를 함께 해석하는 근거로 사용.
[Fed Kitsul·Lang·Samadi(2023)](https://www.federalreserve.gov/econres/notes/feds-notes/monetary-policy-tightening-and-debt-servicing-costs-of-nonfinancial-companies-20231201.html) I–IV절 관련 본문: “increased at a much slower rate”. 기존부채 비용 전달이 신규금리보다 느릴 수 있다는 배경, IV절의 book value of assets 통제는 규모 고려의 참고다. 우리 연도차이·단장기항목 대비·초기영업이익률 추가의 정확한 사양은 연구자 선택이다. 해외 기업 은행차입·현금 지표를 사용한 연구이며 우리 산업패널을 검증한 것은 아니다.

재현: Python3.12 및 requirements.txt, 이 노트북 폴더를 작업경로로 inputs와 함께 전체 셀 실행. 동결된 mechanism_v1 패널부터 새 사전변수·모형·검정 계산. 최초 원자료 구성은 이전 노트북 참조.


In [1]:
# 사전 재무변수 구성
from pathlib import Path
import json,hashlib
import numpy as np,pandas as pd,statsmodels.api as sm
from scipy import stats
from statsmodels.stats.multitest import multipletests
from wildboottest.wildboottest import wildboottest
from IPython.display import display
BASE = Path.cwd()
(BASE / 'tables').mkdir(exist_ok=True)
def save(x, n):
    x.to_csv(BASE / 'tables' / f'{n}.csv', index=False, encoding='utf-8-sig')
for m in json.loads((BASE / 'inputs/manifest.json').read_text(encoding='utf-8')):
    assert hashlib.sha256((BASE / 'inputs' / m['file']).read_bytes()).hexdigest() == m['sha256']
t = pd.read_csv(BASE / 'inputs/panel.csv')
assert len(t) == 240 and t.code.nunique() == 40
assert not t.duplicated(['code','year']).any()
t['SDa'] = 100 * t.short / t['자산총계']
t['LDa'] = 100 * (t['장기차입'] + t['장기사채']) / t['자산총계']
t['Za'] = np.log(t['자산총계'])
a=t[t.year.between(2018,2020)].groupby('code')[['SDa','LDa','Za','margin']].mean().rename(columns={'SDa':'SD','LDa':'LD','Za':'Z0','margin':'P0'})
t = t.merge(a, on='code', validate='many_to_one')
np.testing.assert_allclose(t.SD + t.LD, t.Q, atol=1e-10)
for v in ['SD', 'LD', 'Z0', 'P0']:
    t[v + 'P'] = t[v] * t.Post
    t[v + 'T'] = t[v] * (t.year == 2021)
for v in ['Q', 'S', 'L']:
    for yr in [2021, 2022, 2023]:
        t[v + str(yr)] = t[v] * (t.year == yr)
assert np.isfinite(t[['SD','LD','Z0','P0','cost','icr']]).all().all()
save(t, 'analysis_panel')
save(a.reset_index(), 'added_pre_variables')


### 이 셀에서 준비한 자료

이 셀에는 수치 출력이 없다. 40개 업종 240행을 확인하고 추가 검정에 쓸 변수를 저장하는 단계다.

In [2]:
# 질문별 모형 추정
def design(d,terms):
    X=pd.concat([pd.DataFrame({'const':np.ones(len(d))},index=d.index),d[terms],pd.get_dummies(d.code,prefix='i',drop_first=True,dtype=float),pd.get_dummies(d.year,prefix='y',drop_first=True,dtype=float)],axis=1).astype(float)
    assert np.linalg.matrix_rank(X)==X.shape[1]
    return X
fits = {}
rows = []
def fit(label, d, terms, y='Y'):
    X = design(d, terms)
    model = sm.OLS(d[y].astype(float), X)
    res = model.fit(cov_type='cluster', cov_kwds={'groups': d.code, 'use_correction': True, 'df_correction': True}, use_t=True)
    G = d.code.nunique()
    q = stats.t.ppf(0.975, G - 1)
    for term in terms:
        b = float(res.params[term])
        se = float(res.bse[term])
        rows.append(dict(model=label, outcome=y, term=term, beta=b, se=se, lo=b - q * se, hi=b + q * se, p=float(2 * stats.t.sf(abs(b / se), G - 1)), N=len(d), G=G))
    fits[label] = (d.copy(), terms, model, res)
    return res

base=['QP','SP','LP','QT','ST','LT']
specs={'N1':['SDP','LDP','LP','SDT','LDT','LT'],'N2':[v+str(yr) for v in ['Q','S','L'] for yr in [2021,2022,2023]],'N3':base+['P0P','Z0P','P0T','Z0T']}
for y in ['cost', 'icr']:
    fit('Base_' + y, t, base, y)
    for name, terms in specs.items():
        fit(name + '_' + y, t, terms, y)
coef = pd.DataFrame(rows)
save(coef, 'all_coefficients')

# 계수 간 차이 검정
targets = []
bootmodels = {}
def test(label,a,b=None):
 dd,terms,model,res=fits[label];names=model.exog_names;R=np.zeros(len(names));R[names.index(a)]=1
 if b is not None:R[names.index(b)]=-1
 beta=float(R@res.params.to_numpy());se=float(np.sqrt(R@res.cov_params().to_numpy()@R));q=stats.t.ppf(.975,39)
 name=label+'__'+a+('_minus_'+b if b else '')
 targets.append({'test':name,'model':label,'outcome':label.split('_')[1],'contrast':a+(' - '+b if b else ''),'beta':beta,'se':se,'lo':beta-q*se,'hi':beta+q*se,'p':2*stats.t.sf(abs(beta/se),39),'N':len(dd),'G':40})
 xx=pd.DataFrame(model.exog,columns=names)
 if b is not None:
  # beta_a=delta+beta_b: target열=Xa, nuisance열=Xa+Xb인 동등 재매개화
  xx[b]=xx[a]+xx[b]
 xx=xx.rename(columns={a:'target'});mm=sm.OLS(model.endog,xx)
 np.testing.assert_allclose(mm.fit().params['target'],beta,atol=1e-9)
 rr=mm.fit(cov_type='cluster',cov_kwds={'groups':dd.code,'use_correction':True,'df_correction':True},use_t=True)
 np.testing.assert_allclose(rr.bse['target'],se,atol=1e-8)
 bootmodels[name]=(mm,dd.code)
for y in ['cost', 'icr']:
    test('N1_' + y, 'SDP', 'LDP')
    test('N2_' + y, 'S2023', 'S2022')
    test('N2_' + y, 'Q2023', 'Q2022')
    test('N3_' + y, 'SP')
tests = pd.DataFrame(targets)
tests['p_holm8'] = multipletests(tests.p, method='holm')[1]
save(tests, 'target_tests')
display(tests)
print('All contrast coefficients and CR1 standard errors verified by reparameterization')


                         test    model outcome  ...    N   G   p_holm8
0      N1_cost__SDP_minus_LDP  N1_cost    cost  ...  240  40  1.000000
1  N2_cost__S2023_minus_S2022  N2_cost    cost  ...  240  40  0.987357
2  N2_cost__Q2023_minus_Q2022  N2_cost    cost  ...  240  40  1.000000
3                 N3_cost__SP  N3_cost    cost  ...  240  40  1.000000
4       N1_icr__SDP_minus_LDP   N1_icr     icr  ...  240  40  1.000000
5   N2_icr__S2023_minus_S2022   N2_icr     icr  ...  240  40  1.000000
6   N2_icr__Q2023_minus_Q2022   N2_icr     icr  ...  240  40  0.286125
7                  N3_icr__SP   N3_icr     icr  ...  240  40  1.000000

[8 rows x 12 columns]
All contrast coefficients and CR1 standard errors verified by reparameterization


### 출력에서 읽을 수 있는 것

N1의 비용률 p=0.422, ICR p=0.612는 단기차입 계수와 장기차입 계수의 차이에 대한 값이다. 둘 사이의 차이를 정밀하게 확인하지 못했다.

- 단기·장기차입, 연도 차이, 초기 수익성과 규모를 고려한 모형을 각각 추정했다.
- 두 계수가 같다는 가설을 직접 검정했다. 단기와 장기의 차이는 비용률 p=0.422, ICR p=0.612였다. 한쪽만 유의한지를 보고 차이가 있다고 판정하는 방식보다 질문에 직접 답한다.

In [3]:
# 재표집과 다중검정 비교
boot=[]
for name, (model, groups) in bootmodels.items():
    xx = pd.DataFrame(model.exog, columns=model.exog_names)
    yy = pd.Series(model.endog)
    gg = pd.Series(np.asarray(groups))
    xx = xx - xx.groupby(gg).transform('mean')
    yy = yy - yy.groupby(gg).transform('mean')
    xx = xx.loc[:, np.max(abs(xx), axis=0) > 1e-10]
    assert np.linalg.matrix_rank(xx) == xx.shape[1]
    mm = sm.OLS(yy, xx)
    np.testing.assert_allclose(mm.fit().params.target, model.fit().params.target, atol=1e-08)
    r = wildboottest(mm, param='target', cluster=pd.factorize(gg)[0], B=9999, seed=20261103, bootstrap_type='31', impose_null=True, weights_type='rademacher', show=False)
    boot.append({'test': name, 'p_boot': float(r['p-value'].iloc[0]), 'B': 9999, 'seed': 20261103})
boot = pd.DataFrame(boot)
boot['p_boot_holm8'] = multipletests(boot.p_boot, method='holm')[1]
save(boot, 'bootstrap')
combined = tests.merge(boot, on='test', validate='one_to_one')
save(combined, 'combined_tests')
display(combined)


                         test    model  ...      seed  p_boot_holm8
0      N1_cost__SDP_minus_LDP  N1_cost  ...  20261103      1.000000
1  N2_cost__S2023_minus_S2022  N2_cost  ...  20261103      1.000000
2  N2_cost__Q2023_minus_Q2022  N2_cost  ...  20261103      1.000000
3                 N3_cost__SP  N3_cost  ...  20261103      1.000000
4       N1_icr__SDP_minus_LDP   N1_icr  ...  20261103      1.000000
5   N2_icr__S2023_minus_S2022   N2_icr  ...  20261103      1.000000
6   N2_icr__Q2023_minus_Q2022   N2_icr  ...  20261103      0.813681
7                  N3_icr__SP   N3_icr  ...  20261103      1.000000

[8 rows x 16 columns]


### 출력에서 읽을 수 있는 것

N2의 ICR 연도 차이는 일반 p=0.036에서 부트스트랩 p=0.102로 바뀌었다. 검정 방식에 민감해 두 연도의 관계가 확실히 다르다고 말하기 어렵다.

## 실행 결과와 해석

8개 핵심검정 모두 공개했다. N1 단기−장기 대비: 비용률 p=.4223/부트스트랩 .4594, ICR .6116/.6546. N2 Q의 ICR 2023−2022 차이는 +1.8200, 일반 p=.0358이지만 부트스트랩 .1017, Holm .2861, 부트스트랩 Holm .8137이다. 이 양의 연도차이는 원래 단기차입 악화 가설의 확증이 아니다. N3 S×Post는 비용률 p=.2829, ICR .3885로 초기 수익성과 업종 자산 규모의 통제를 추가해도 증거가 뚜렷해지지 않았다.

비유의는 무관계나 동등성 증명이 아니다. 공식 차입항목·집계 정의가 미확인인 대용률 해석을 유지한다. 새 결과에 따라 표본·기간·이상치 기준을 재선택하지 않았다. 전체 설명·기존 결과·필요자료는 함께 제공한 HTML을 참조한다.